# Datamine TRUETHK Process Example

This notebook demonstrates how to configure and run the **`truethk`** process wrapper in `dmstudio`.

## Process Description

## TRUETHK

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

For tabular ore bodies, based on a set of static drillhole sample data, composites are commonly created across the defined ore boundaries.

These composites can be defined by either lithological codes, or by selection and assignment of codes based on physically defined limits perimeters and/or wireframes. For subsequent analysis and reserve evaluation, a common requirement is the true width of each ore composite.

Sometimes, the horizontal and/or vertical thicknesses are also required. The **TRUETHK** process calculates these true thickness values, based on the supplied sample lengths and orientations, and the supplied ore body dip and dip direction across which the samples have been taken.

The input file to the process is a standard desurveyed sample file, which must contain at least the fields **LENGTH** , **A0** and **B0**. Compulsory input parameters are the ore body dip and dip direction. The output file created will essentially be a copy of the input sample file, but with the additional fields **TRUETHK** , **VERTHK** , **HORTHK** and **TRUEDIP**.

### Input Files:

* **in** (Sample Data):
  Sample/composite data file.
  Required=Yes

### Output Files:

* **out** (Sample data):
  Sample/composite data with calculated true, vertical and horizontal thickness fields.
  Required=Yes

### Fields:

### Parameters:

* **dip**:
  Ore body dip value.
  Range=0,90
  Values=
  Default=0
  Required=Yes

* **dipdirn**:
  Ore body dip direction.
  Range=0,360
  Values=
  Default=0
  Required=Yes

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('truethk')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute truethk
print("Running truethk...")
dm_cmd.truethk(
    in_i='t_assays',  # required
    out_o='t_truethk_out',  # required
    dip_p='required_val',  # required
    dipdirn_p='required_val',  # required
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("truethk execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_truethk_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")